In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files (they are in cwd)
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

def compact_time_range(df, ts_col='timestamp'):
    if ts_col not in df.columns or df[ts_col].dropna().empty:
        return {"min_epoch": None, "max_epoch": None, "min_ts_local": None, "max_ts_local": None}
    mn = int(df[ts_col].min())
    mx = int(df[ts_col].max())
    mn_dt = pd.to_datetime(mn, unit='s', utc=True).tz_convert(tz)
    mx_dt = pd.to_datetime(mx, unit='s', utc=True).tz_convert(tz)
    return {
        "min_epoch": mn,
        "max_epoch": mx,
        "min_ts_local": mn_dt.strftime('%Y-%m-%d %H:%M:%S %Z'),
        "max_ts_local": mx_dt.strftime('%Y-%m-%d %H:%M:%S %Z'),
    }

def top_counts(df, group_col, topn=20):
    if group_col not in df.columns:
        return pd.DataFrame(columns=[group_col, 'count'])
    s = df.groupby(group_col).size().sort_values(ascending=False).head(topn).reset_index(name='count')
    return s

# metric aggregates
metric_summary = {
    "total_rows": len(metric),
    "distinct_cmdb_id": int(metric['cmdb_id'].nunique()) if 'cmdb_id' in metric.columns else 0,
    "top20_cmdb_id_by_count": top_counts(metric, 'cmdb_id'),
    "time_range": compact_time_range(metric, 'timestamp'),
    "top20_kpi_name_by_count": top_counts(metric, 'kpi_name'),
    "distinct_kpi_sample_count": int(metric['kpi_name'].nunique()) if 'kpi_name' in metric.columns else 0
}

# trace aggregates
trace_summary = {
    "total_rows": len(trace),
    "distinct_cmdb_id": int(trace['cmdb_id'].nunique()) if 'cmdb_id' in trace.columns else 0,
    "top20_cmdb_id_by_count": top_counts(trace, 'cmdb_id'),
    "time_range": compact_time_range(trace, 'timestamp'),
    "top20_trace_name_by_count": top_counts(trace, 'trace_name'),
    "distinct_trace_name_count": int(trace['trace_name'].nunique()) if 'trace_name' in trace.columns else 0
}

# log aggregates
log_summary = {
    "total_rows": len(log),
    "distinct_cmdb_id": int(log['cmdb_id'].nunique()) if 'cmdb_id' in log.columns else 0,
    "top20_cmdb_id_by_count": top_counts(log, 'cmdb_id'),
    "time_range": compact_time_range(log, 'timestamp'),
    "top20_log_name_by_count": top_counts(log, 'log_name'),
    "distinct_log_name_count": int(log['log_name'].nunique()) if 'log_name' in log.columns else 0
}

# error_logs aggregates
# show up to 20 distinct message samples and their counts (first 20 distinct messages by frequency)
err_msg_counts = top_counts(error_logs, 'message')
error_logs_summary = {
    "total_rows": len(error_logs),
    "distinct_cmdb_id": int(error_logs['cmdb_id'].nunique()) if 'cmdb_id' in error_logs.columns else 0,
    "top20_cmdb_id_by_count": top_counts(error_logs, 'cmdb_id'),
    "time_range": compact_time_range(error_logs, 'timestamp'),
    "top20_message_samples_by_count": err_msg_counts,
    "distinct_message_count": int(error_logs['message'].nunique()) if 'message' in error_logs.columns else 0
}

summary = {
    "metric.csv": metric_summary,
    "trace.csv": trace_summary,
    "log.csv": log_summary,
    "error_logs.csv": error_logs_summary
}

summary

```
Out[1]:
```
result = (
    "Summary of telemetry files (compact):\n\n"
    "metric.csv:\n"
    "- Total rows: 144,252\n"
    "- Distinct cmdb_id: 59\n"
    "- Top cmdb_ids (by rows): adservice (10230), adservice2 (9990), frontend-0 (4800), frontend2-0 (4800), frontend-1 (4680), frontend-2 (4380), checkoutservice-0/1/2 (3780/3540/3540), productcatalogservice-* (3360–3480), ... (top 20 available)\n"
    "- Time range (UTC+8): 2022-03-20 14:30:00 to 2022-03-20 14:59:00\n"
    "- Distinct kpi_name: 1,152; many top KPIs are container.node-5.* metrics (several ~1,050 rows each)\n\n"
    "trace.csv:\n"
    "- Total rows: 36,800\n"
    "- Distinct cmdb_id: 40\n"
    "- Top cmdb_ids (by rows): frontend-2 (2756), frontend-0 (2756), checkoutservice-2 (2744), checkoutservice-0 (2732), frontend-1 (2724), checkoutservice-1 (2724), ... (top 20 available)\n"
    "- Time range (UTC+8): 2022-03-20 14:30:00 to 2022-03-20 14:59:00\n"
    "- Distinct trace_name: 200; top trace_names are trace.from_frontend-2.* and trace.from_frontend-0.* (~629 rows each)\n\n"
    "log.csv:\n"
    "- Total rows: 1,812\n"
    "- Distinct cmdb_id: 31\n"
    "- Top cmdb_ids (by rows): many services/pods with 60 rows each (adservice-0/1/2, cartservice-0/1/2, checkoutservice-0/1/2, etc.)\n"
    "- Time range (UTC+8): 2022-03-20 14:30:00 to 2022-03-20 14:59:00\n"
    "- Distinct log_name: 2 (log.error_count: 906 rows, log.row_count: 906 rows)\n\n"
    "error_logs.csv:\n"
    "- Total rows: 1,915\n"
    "- Distinct cmdb_id: 3 (frontend-2: 692, frontend-0: 681, frontend-1: 542)\n"
    "- Time range (UTC+8): 2022-03-20 14:30:00 to 2022-03-20 14:59:58\n"
    "- Distinct messages: 1 (single message sample repeated for all 1,915 rows: \"severity: warning, message: failed to retrieve...\")\n\n"
    "Overall observation: metric and trace data are large and centered around frontend and adservice-related components; logs are small and evenly distributed across pods; all error log entries come from frontends and are the same message repeated across the interval."
)
result

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'total_rows': 144252, 'distinct_cmdb_id': 59, 'top20_cmdb_id_by_count':                      cmdb_id  count
0                  adservice  10230
1                 adservice2   9990
2                 frontend-0   4800
3                frontend2-0   4800
4                 frontend-1   4680
5                 frontend-2   4380
6          checkoutservice-0   3780
7          checkoutservice-1   3540
8          checkoutservice-2   3540
9    productcatalogservice-0   3480
10   productcatalogservice-2   3480
11   productcatalogservice-1   3360
12        checkoutservice2-0   3300
13  productcatalogservice2-0   2880
14             cartservice-1   2760
15             cartservice-0   2640
16            cartservice2-0   2640
17              adservice2-0   2580
18               adservice-0   2580
19             cartservice-2   2520, 'time_range': {'min_epoch': 1647757800, 'max_epoch': 1647759540, 'min_ts_local': '2022-03-20 14:30:00 CST', 'max_ts_local': '2022-03-20 14:59:00 CST'}, 'top20_kpi_name_by_count':                                              kpi_name  count
0   container.node-5.container_network_transmit_pa...   1050
1   container.node-5.container_network_transmit_pa...   1050
2   container.node-5.container_network_transmit_er...   1050
3   container.node-5.container_network_transmit_MB...   1050
4   container.node-5.container_network_receive_pac...   1050
5   container.node-5.container_network_receive_pac...   1050
6   container.node-5.container_network_receive_err...   1050
7   container.node-5.container_network_receive_MB....   1050
8             container.node-5.container_memory_cache    990
9           container.node-5.container_memory_failcnt    990
10  container.node-5.container_memory_failures.con...    990
11  container.node-5.container_memory_failures.con...    990
12         container.node-5.container_spec_cpu_shares    990
13  container.node-5.container_fs_sector_reads./de...    990
14  container.node-5.container_fs_io_time_seconds....    990
15  container.node-5.container_fs_reads_merged./de...    990
16  container.node-5.container_fs_inodes_free./dev...    990
17  container.node-5.container_memory_failures.hie...    990
18      container.node-5.container_memory_mapped_file    990
19     container.node-5.container_memory_max_usage_MB    990, 'distinct_kpi_sample_count': 1152}, 'trace.csv': {'total_rows': 36800, 'distinct_cmdb_id': 40, 'top20_cmdb_id_by_count':                     cmdb_id  count
0                frontend-2   2756
1                frontend-0   2756
2         checkoutservice-2   2744
3         checkoutservice-0   2732
4                frontend-1   2724
5         checkoutservice-1   2724
6   productcatalogservice-1   1200
7   productcatalogservice-2   1200
8   productcatalogservice-0   1200
9               frontend2-0   1080
10       checkoutservice2-0   1080
11  recommendationservice-0    840
12  recommendationservice-2    840
13            cartservice-0    840
14            cartservice-2    840
15            cartservice-1    840
16  recommendationservice-1    840
17        shippingservice-1    720
18        shippingservice-0    720
19        shippingservice-2    720, 'time_range': {'min_epoch': 1647757800, 'max_epoch': 1647759540, 'min_ts_local': '2022-03-20 14:30:00 CST', 'max_ts_local': '2022-03-20 14:59:00 CST'}, 'top20_trace_name_by_count':                                     trace_name  count
0          trace.from_frontend-2.duration_mean    629
1           trace.from_frontend-2.duration_p95    629
2             trace.from_frontend-2.error_rate    629
3              trace.from_frontend-2.row_count    629
4          trace.from_frontend-0.duration_mean    629
5           trace.from_frontend-0.duration_p95    629
6             trace.from_frontend-0.error_rate    629
7              trace.from_frontend-0.row_count    629
8              trace.from_frontend-1.row_count    621
9             trace.from_frontend-1.error_rate    621
10          trace.from_frontend-1.duration_p95    621
11         trace.from_frontend-1.duration_mean    621
12      trace.from_checkoutservice-2.row_count    539
13     trace.from_checkoutservice-2.error_rate    539
14   trace.from_checkoutservice-2.duration_p95    539
15  trace.from_checkoutservice-2.duration_mean    539
16      trace.from_checkoutservice-1.row_count    536
17     trace.from_checkoutservice-1.error_rate    536
18   trace.from_checkoutservice-1.duration_p95    536
19  trace.from_checkoutservice-1.duration_mean    536, 'distinct_trace_name_count': 200}, 'log.csv': {'total_rows': 1812, 'distinct_cmdb_id': 31, 'top20_cmdb_id_by_count':               cmdb_id  count
0         adservice-0     60
1         adservice-1     60
2         adservice-2     60
3       cartservice-0     60
4       cartservice-1     60
5       cartservice-2     60
6   checkoutservice-0     60
7   checkoutservice-1     60
8   checkoutservice-2     60
9   currencyservice-0     60
10  currencyservice-1     60
11  currencyservice-2     60
12     emailservice-0     60
13     emailservice-1     60
14     emailservice-2     60
15         frontend-0     60
16         frontend-1     60
17         frontend-2     60
18   paymentservice-0     60
19   paymentservice-1     60, 'time_range': {'min_epoch': 1647757800, 'max_epoch': 1647759540, 'min_ts_local': '2022-03-20 14:30:00 CST', 'max_ts_local': '2022-03-20 14:59:00 CST'}, 'top20_log_name_by_count':           log_name  count
0  log.error_count    906
1    log.row_count    906, 'distinct_log_name_count': 2}, 'error_logs.csv': {'total_rows': 1915, 'distinct_cmdb_id': 3, 'top20_cmdb_id_by_count':       cmdb_id  count
0  frontend-2    692
1  frontend-0    681
2  frontend-1    542, 'time_range': {'min_epoch': 1647757800, 'max_epoch': 1647759598, 'min_ts_local': '2022-03-20 14:30:00 CST', 'max_ts_local': '2022-03-20 14:59:58 CST'}, 'top20_message_samples_by_count':                                              message  count
0  severity: warning, message: failed to retrieve...   1915, 'distinct_message_count': 1}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# reuse existing 'metric' DataFrame from previous execution; if not present, load it
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

components = ['frontend-0','frontend-1','frontend-2','frontend2-0']

# Ensure numeric
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce').astype('Int64')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using full series (no time filter)
mask_components = metric['cmdb_id'].isin(components)
grp_all = metric[mask_components].groupby(['cmdb_id','kpi_name'])['value']
global_p95 = grp_all.quantile(0.95).rename('global_p95')
global_p05 = grp_all.quantile(0.05).rename('global_p5')
thresholds = pd.concat([global_p95, global_p05], axis=1).reset_index()

# 2) Filter to incident window in UTC+8: 2022-03-20 14:30:00 to 15:00:00 (use start inclusive, end exclusive)
start_epoch = int(pd.Timestamp('2022-03-20 14:30:00', tz=tz).timestamp())
end_epoch = int(pd.Timestamp('2022-03-20 15:00:00', tz=tz).timestamp())
window_mask = mask_components & (metric['timestamp'] >= start_epoch) & (metric['timestamp'] < end_epoch)
window_df = metric[window_mask].copy()

# Merge thresholds into window rows to evaluate anomalies
window_with_thr = window_df.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Determine anomaly per row (value >= global_p95 OR value <= global_p5)
window_with_thr['is_anom'] = ((window_with_thr['value'] >= window_with_thr['global_p95']) |
                              (window_with_thr['value'] <= window_with_thr['global_p5'])).fillna(False)

# Aggregate in-window stats per component-kpi
agg_funcs = {
    'value': ['count', 'max', 'min'],
    'is_anom': 'sum',
    'timestamp': lambda s: s.loc[window_with_thr.loc[s.index, 'is_anom']].min() if s.loc[window_with_thr.loc[s.index, 'is_anom']].any() else pd.NA
}
grouped = window_with_thr.groupby(['cmdb_id','kpi_name']).agg({
    'value': ['count','max','min'],
    'is_anom': 'sum',
    'timestamp': lambda s: (s[window_with_thr.loc[s.index,'is_anom']].min() if s[window_with_thr.loc[s.index,'is_anom']].any() else pd.NA)
})

# Normalize multiindex columns
grouped.columns = ['points_in_window','max_in_window','min_in_window','anomaly_count','earliest_anomaly_ts']
grouped = grouped.reset_index()

# Merge with thresholds to ensure all (component,kpi) pairs are present (including those with 0 in-window points)
result = thresholds.merge(grouped, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for groups with no window data
result['points_in_window'] = result['points_in_window'].fillna(0).astype(int)
result['anomaly_count'] = result['anomaly_count'].fillna(0).astype(int)
# max/min set to NA if no points
result['max_in_window'] = result['max_in_window']
result['min_in_window'] = result['min_in_window']

# Convert earliest_anomaly_ts from epoch to local tz string, keep NA as None
def epoch_to_local_str(x):
    if pd.isna(x):
        return None
    try:
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return None

result['earliest_anomaly_ts'] = result['earliest_anomaly_ts'].apply(epoch_to_local_str)

# Keep only required columns and order
out_cols = ['cmdb_id','kpi_name','global_p95','global_p5','points_in_window','anomaly_count','earliest_anomaly_ts','max_in_window','min_in_window']
result = result[out_cols]

# Sort by anomaly_count desc and limit to top 20
result_table = result.sort_values(by='anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Round numeric thresholds for compactness
result_table['global_p95'] = result_table['global_p95'].round(6)
result_table['global_p5'] = result_table['global_p5'].round(6)
result_table['max_in_window'] = result_table['max_in_window'].round(6)
result_table['min_in_window'] = result_table['min_in_window'].round(6)

# Total anomaly points across these frontend components in the window
total_anomalies = int(result['anomaly_count'].sum())

summary_line = f"Total anomaly points across {components} in window: {total_anomalies}"

# Display compact table and summary
result_table, total_anomalies, summary_line

```
Out[2]:
```
Summary (plain English):

- What I computed: for frontend-0, frontend-1, frontend-2, frontend2-0 I calculated global P95 and P5 for each component–KPI using the full series, then inspected the incident window (2022-03-20 14:30:00 to 15:00:00 UTC+8) and counted in-window points, anomaly points (value >= global_P95 or <= global_P5), earliest in-window anomaly, and in-window max/min.

- Main findings (compact):
  - Total anomaly points across the four frontends in the 30-minute window: 11,270.
  - The Top results (top 20 component–KPI rows by anomaly count) each show 30 points in-window and 30 anomalies. For these top rows:
    - global_p95 and global_p5 are frequently 0.0, so every in-window value of 0.0 is flagged as anomalous.
    - earliest anomaly timestamp for the top rows is 2022-03-20 14:30:00 (UTC+8).
    - in-window max and min are typically 0.0 (many metrics) — a few constant metrics (e.g., file_descriptors) have global_p95 = global_p5 = 22.0 and in-window values 22.0 (also flagged).
  - The anomaly pattern indicates many KPIs are constant at or near zero (or a single constant value) so the P5/P95 thresholds equal those constants, causing all in-window points to meet the anomaly condition.

- Interpretation: a large number of metrics for these frontends have zero or constant values historically, so by the percentile rule used (value >= P95 or <= P5) nearly every point in the window is classified as anomalous. Earliest anomalies appear at the start of the window (14:30).

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                           kpi_name  global_p95  global_p5  points_in_window  anomaly_count      earliest_anomaly_ts  max_in_window  min_in_window
0   frontend2-0  mesh.source.frontend2.unknown.istio_response_m...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
1   frontend2-0  mesh.source.frontend2.unknown.istio_response_b...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
2    frontend-0  container.node-5.container_cpu_cfs_throttled_p...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
3    frontend-0  container.node-5.container_cpu_cfs_throttled_s...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
4    frontend-0    container.node-5.container_cpu_load_average_10s         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
5   frontend2-0  mesh.source.frontend2.unknown.istio_requests.g...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
6   frontend2-0  mesh.source.frontend2.unknown.istio_request_me...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
7   frontend2-0  mesh.source.frontend2.unknown.istio_request_du...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
8    frontend-0        container.node-5.container_file_descriptors        22.0       22.0                30             30  2022-03-20 14:30:00 CST           22.0           22.0
9    frontend-0     container.node-5.container_fs_inodes./dev/vda1         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
10  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
11  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
12  frontend2-0  mesh.source.frontend2.cartservice2.istio_respo...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
13  frontend2-0  mesh.source.frontend2.cartservice2.istio_respo...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
14  frontend2-0  mesh.source.frontend2.adservice2.istio_respons...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
15  frontend2-0  mesh.source.frontend2.adservice2.istio_respons...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
16  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
17  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
18  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0
19  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...         0.0        0.0                30             30  2022-03-20 14:30:00 CST            0.0            0.0, 11270, "Total anomaly points across ['frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0'] in window: 11270")```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# reuse 'trace' if present, otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# Ensure types
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace['timestamp'] = pd.to_numeric(trace['timestamp'], errors='coerce').astype('Int64')

# Targets
components = ['frontend-0','frontend-1','frontend-2','frontend2-0']
patterns = ['error_rate','row_count','duration']
pattern_re = '|'.join(patterns)

# Filter rows for relevant components and trace_name patterns (case-insensitive)
mask_components = trace['cmdb_id'].isin(components)
name_mask = trace['trace_name'].str.contains(pattern_re, case=False, na=False)
relevant_df = trace[mask_components & name_mask].copy()

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using the full available series (no time filter beyond components+name)
grp = relevant_df.groupby(['cmdb_id','trace_name'])['value']
global_p95 = grp.quantile(0.95).rename('global_p95')
global_p5  = grp.quantile(0.05).rename('global_p5')
thresholds = pd.concat([global_p95, global_p5], axis=1).reset_index()

# Incident window (inclusive start, exclusive end) in UTC+8
start_epoch = int(pd.Timestamp('2022-03-20 14:30:00', tz=tz).timestamp())
end_epoch   = int(pd.Timestamp('2022-03-20 15:00:00', tz=tz).timestamp())

# 2) Filter trace to the window for the same components+patterns
window_df = relevant_df[(relevant_df['timestamp'] >= start_epoch) & (relevant_df['timestamp'] < end_epoch)].copy()

# Merge thresholds into window rows
win = window_df.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

# Compute anomaly flags
win['anomaly_high'] = (win['value'] >= win['global_p95']).fillna(False)
win['anomaly_low']  = (win['value'] <= win['global_p5']).fillna(False)

# Aggregates in-window
points = win.groupby(['cmdb_id','trace_name'])['value'].count().rename('points_in_window')
max_in = win.groupby(['cmdb_id','trace_name'])['value'].max().rename('max_in_window')
min_in = win.groupby(['cmdb_id','trace_name'])['value'].min().rename('min_in_window')
an_high = win.groupby(['cmdb_id','trace_name'])['anomaly_high'].sum().rename('anomalies_high')
an_low  = win.groupby(['cmdb_id','trace_name'])['anomaly_low'].sum().rename('anomalies_low')

# earliest anomaly timestamps (separately for high and low)
high_ts = win[win['anomaly_high']].groupby(['cmdb_id','trace_name'])['timestamp'].min().rename('earliest_anomaly_high_ts')
low_ts  = win[win['anomaly_low']].groupby(['cmdb_id','trace_name'])['timestamp'].min().rename('earliest_anomaly_low_ts')

# Combine aggregates
agg = thresholds.set_index(['cmdb_id','trace_name']).join(points).join(max_in).join(min_in).join(an_high).join(an_low).reset_index()

# Merge earliest timestamps (may be missing)
agg = agg.merge(high_ts.reset_index(), on=['cmdb_id','trace_name'], how='left')
agg = agg.merge(low_ts.reset_index(), on=['cmdb_id','trace_name'], how='left')

# Fill missing numeric window aggregates with zeros or NaN for max/min
agg['points_in_window'] = agg['points_in_window'].fillna(0).astype(int)
agg['anomalies_high'] = agg['anomalies_high'].fillna(0).astype(int)
agg['anomalies_low'] = agg['anomalies_low'].fillna(0).astype(int)

# Convert earliest anomaly epochs to localized strings
def epoch_to_local_str(x):
    if pd.isna(x):
        return None
    try:
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return None

agg['earliest_anomaly_high_ts'] = agg['earliest_anomaly_high_ts'].apply(epoch_to_local_str)
agg['earliest_anomaly_low_ts']  = agg['earliest_anomaly_low_ts'].apply(epoch_to_local_str)

# suggested_direction
def suggest(row):
    hi = int(row['anomalies_high'])
    lo = int(row['anomalies_low'])
    if hi>lo:
        return 'high'
    if lo>hi:
        return 'low'
    if hi==lo and hi>0:
        return 'both'
    return 'none'

agg['suggested_direction'] = agg.apply(suggest, axis=1)

# Prepare final table and formatting
cols = ['cmdb_id','trace_name','global_p95','global_p5','points_in_window',
        'anomalies_high','anomalies_low','earliest_anomaly_high_ts','earliest_anomaly_low_ts',
        'max_in_window','min_in_window','suggested_direction']
result = agg[cols].copy()
result['global_p95'] = result['global_p95'].round(6)
result['global_p5'] = result['global_p5'].round(6)
result['max_in_window'] = result['max_in_window'].round(6)
result['min_in_window'] = result['min_in_window'].round(6)

# Sort by total anomalies desc and take top 20
result['total_anoms'] = result['anomalies_high'] + result['anomalies_low']
result_table = result.sort_values(by='total_anoms', ascending=False).head(20).drop(columns=['total_anoms']).reset_index(drop=True)

# Summary: total anomaly points and earliest anomaly overall
total_anomalies = int(result['anomalies_high'].sum() + result['anomalies_low'].sum())
all_earliest = pd.Series(list(result['earliest_anomaly_high_ts'].dropna()) + list(result['earliest_anomaly_low_ts'].dropna()))
if not all_earliest.empty:
    earliest_overall = pd.to_datetime(all_earliest).min().tz_localize(None).tz_localize('UTC').tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    earliest_overall = None

summary_line = f"Total anomaly points in window (frontends, selected traces): {total_anomalies}; earliest anomaly: {earliest_overall}"

# Compact outputs
result_table, total_anomalies, earliest_overall, summary_line

```
Out[3]:
```
Summary (plain English):

- What I did: for frontend-0, frontend-1, frontend-2, frontend2-0 and trace names containing error_rate, row_count or duration, I computed global P95/P5 from the full series, then inspected the incident window 2022-03-20 14:30–15:00 (UTC+8) and counted high/low anomalies per (cmdb_id, trace_name).

- Key results:
  - Total anomaly points in the window across these frontend-origin traces: 6,614.
  - Earliest reported anomaly (from the computed results): 2022-03-20 22:30:00 CST.
  - Top rows (examples from the top 20): each shows 30 points in-window and 30 high + 30 low anomalies, global_p95 = 0.0 and global_p5 = 0.0, earliest anomaly timestamps at 2022-03-20 14:30:00 CST, max_in_window = 0.0, min_in_window = 0.0. The suggested_direction for these rows is "both" (anomalies_high == anomalies_low > 0).

- Interpretation: most selected trace metrics are constant at zero historically (global P95 and P5 = 0.0). Because the anomaly rule flags values >= P95 or <= P5, every in-window zero-valued point is marked as both a high and low anomaly, producing large anomaly counts. This pattern suggests the percentile-based rule is producing many false/ambiguous positives when the metric distribution is degenerate (constant zeros).

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                    trace_name  global_p95  global_p5  points_in_window  anomalies_high  anomalies_low earliest_anomaly_high_ts  earliest_anomaly_low_ts  max_in_window  min_in_window suggested_direction
0   frontend2-0        trace.to_shippingservice2-0.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
1    frontend-0                    trace.from_root.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
2   frontend2-0  trace.to_recommendationservice2-0.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
3    frontend-0                         trace.self.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
4    frontend-0               trace.to_adservice-0.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
5   frontend2-0  trace.to_productcatalogservice2-0.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
6    frontend-2   trace.to_productcatalogservice-0.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
7    frontend-2         trace.to_currencyservice-2.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
8    frontend-2   trace.to_recommendationservice-0.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
9    frontend-2   trace.to_recommendationservice-1.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
10   frontend-2         trace.to_shippingservice-0.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
11   frontend-2   trace.to_recommendationservice-2.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
12   frontend-2   trace.to_productcatalogservice-2.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
13   frontend-2   trace.to_productcatalogservice-1.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
14   frontend-0         trace.to_currencyservice-1.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
15   frontend-2         trace.to_currencyservice-0.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
16   frontend-2             trace.to_cartservice-2.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
17   frontend-2         trace.to_checkoutservice-2.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
18   frontend-2         trace.to_checkoutservice-1.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both
19   frontend-2         trace.to_currencyservice-1.error_rate         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0                both, 6614, '2022-03-20 22:30:00 CST', 'Total anomaly points in window (frontends, selected traces): 6614; earliest anomaly: 2022-03-20 22:30:00 CST')```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse existing dataframes if present, otherwise load
try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')
try:
    log
except NameError:
    log = pd.read_csv('log.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp and numeric types
error_logs['timestamp'] = pd.to_numeric(error_logs['timestamp'], errors='coerce').astype('Int64')
log['timestamp'] = pd.to_numeric(log['timestamp'], errors='coerce').astype('Int64')
log['value'] = pd.to_numeric(log['value'], errors='coerce')

# Incident window in UTC+8
start_epoch = int(pd.Timestamp('2022-03-20 14:30:00', tz=tz).timestamp())
end_epoch   = int(pd.Timestamp('2022-03-20 15:00:00', tz=tz).timestamp())

frontends = ['frontend-0','frontend-1','frontend-2']

# 1) error_logs: filter by window and frontends
err_win = error_logs[
    (error_logs['cmdb_id'].isin(frontends)) &
    (error_logs['timestamp'] >= start_epoch) &
    (error_logs['timestamp'] < end_epoch)
].copy()

# Helper to convert epoch to local string
def epoch_to_local_str(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# Aggregate per frontend
if not err_win.empty:
    grp_err_counts = err_win.groupby('cmdb_id').agg(
        total_error_logs_in_window=('timestamp','count'),
        earliest_error_ts_in_window=('timestamp','min'),
        latest_error_ts_in_window=('timestamp','max')
    ).reset_index()
    # convert timestamps
    grp_err_counts['earliest_error_ts_in_window'] = grp_err_counts['earliest_error_ts_in_window'].apply(epoch_to_local_str)
    grp_err_counts['latest_error_ts_in_window'] = grp_err_counts['latest_error_ts_in_window'].apply(epoch_to_local_str)

    # Top 3 distinct message samples with counts per frontend
    msg_counts = (err_win.groupby(['cmdb_id','message'])
                      .size()
                      .reset_index(name='count'))
    # For each cmdb_id, get top 3 messages and format as "msg (n)"
    def top_msgs_for_cmdb(cmdb):
        sub = msg_counts[msg_counts['cmdb_id']==cmdb].sort_values('count', ascending=False).head(3)
        parts = [f"{m} ({c})" for m,c in zip(sub['message'], sub['count'])]
        return "; ".join(parts) if parts else None

    grp_err_counts['top_messages'] = grp_err_counts['cmdb_id'].apply(top_msgs_for_cmdb)
    error_summary_df = grp_err_counts[['cmdb_id','total_error_logs_in_window','earliest_error_ts_in_window','latest_error_ts_in_window','top_messages']]
else:
    # empty table with required columns
    error_summary_df = pd.DataFrame(columns=['cmdb_id','total_error_logs_in_window','earliest_error_ts_in_window','latest_error_ts_in_window','top_messages'])

# 2) log.csv: filter by window and frontends
log_win = log[
    (log['cmdb_id'].isin(frontends)) &
    (log['timestamp'] >= start_epoch) &
    (log['timestamp'] < end_epoch)
].copy()

if not log_win.empty:
    # Aggregates per (cmdb_id, log_name)
    agg_basic = log_win.groupby(['cmdb_id','log_name']).agg(
        total_points_in_window=('value','count'),
        sum_value_in_window=('value','sum'),
        max_in_window=('value','max')
    ).reset_index()

    # earliest timestamp where value > 0 per group
    pos = log_win[log_win['value']>0].copy()
    if not pos.empty:
        earliest_pos = pos.groupby(['cmdb_id','log_name'])['timestamp'].min().reset_index(name='earliest_ts_where_value_gt_0')
    else:
        earliest_pos = pd.DataFrame(columns=['cmdb_id','log_name','earliest_ts_where_value_gt_0'])

    log_summary = agg_basic.merge(earliest_pos, on=['cmdb_id','log_name'], how='left')
    # format earliest_ts_where_value_gt_0
    log_summary['earliest_ts_where_value_gt_0'] = log_summary['earliest_ts_where_value_gt_0'].apply(lambda x: epoch_to_local_str(x) if pd.notna(x) else None)
    # round numeric sums for compactness
    log_summary['sum_value_in_window'] = log_summary['sum_value_in_window'].round(6)
    log_summary['max_in_window'] = log_summary['max_in_window'].round(6)

    # limit to top 20 rows by total_points_in_window desc
    log_summary_df = log_summary.sort_values('total_points_in_window', ascending=False).head(20).reset_index(drop=True)
else:
    log_summary_df = pd.DataFrame(columns=['cmdb_id','log_name','total_points_in_window','sum_value_in_window','earliest_ts_where_value_gt_0','max_in_window'])

# 3) One-line summary: which frontend has highest error_logs count and earliest error timestamp among all frontends
if not error_summary_df.empty:
    top_frontend_row = error_summary_df.sort_values('total_error_logs_in_window', ascending=False).iloc[0]
    frontend_highest_errors = top_frontend_row['cmdb_id']
    highest_error_count = int(top_frontend_row['total_error_logs_in_window'])
    # earliest across all frontends
    earliest_overall_epoch = err_win['timestamp'].min()
    earliest_overall = epoch_to_local_str(earliest_overall_epoch) if pd.notna(earliest_overall_epoch) else None
    summary_line = f"Highest error_logs count: {frontend_highest_errors} ({highest_error_count} errors); earliest error overall: {earliest_overall}"
else:
    frontend_highest_errors = None
    highest_error_count = 0
    earliest_overall = None
    summary_line = "No error logs in window for the specified frontends."

# Display compact results
error_summary_df, log_summary_df, summary_line

```
Out[4]:
```
Summary (plain English, times in UTC+8):

1) error_logs.csv (window 2022-03-20 14:30:00–15:00:00 UTC+8)
- frontend-0: 681 error log entries; earliest 2022-03-20 14:30:00; latest 2022-03-20 14:59:56. Top message sample (up to 3): "severity: warning, message: failed to retrieve..." (681).
- frontend-1: 542 error log entries; earliest 2022-03-20 14:30:02; latest 2022-03-20 14:59:57. Top message: "severity: warning, message: failed to retrieve..." (542).
- frontend-2: 692 error log entries; earliest 2022-03-20 14:30:01; latest 2022-03-20 14:59:58. Top message: "severity: warning, message: failed to retrieve..." (692).

2) log.csv (same window)
- frontend-0:
  - log.error_count: 30 points, sum_value = 681.0, max = 28.0, earliest value>0 at 2022-03-20 14:30:00.
  - log.row_count:   30 points, sum_value = 52,262.0, max = 1,917.0, earliest value>0 at 2022-03-20 14:30:00.
- frontend-1:
  - log.error_count: 30 pts, sum = 542.0, max = 22.0, earliest >0 at 2022-03-20 14:30:00.
  - log.row_count:   30 pts, sum = 42,253.0, max = 1,618.0, earliest >0 at 2022-03-20 14:30:00.
- frontend-2:
  - log.error_count: 30 pts, sum = 692.0, max = 27.0, earliest >0 at 2022-03-20 14:30:00.
  - log.row_count:   30 pts, sum = 53,281.0, max = 1,994.0, earliest >0 at 2022-03-20 14:30:00.

One-line summary:
- Frontend with highest error_logs count in the window: frontend-2 (692 errors). Earliest error across these frontends: 2022-03-20 14:30:00 UTC+8.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_error_logs_in_window earliest_error_ts_in_window latest_error_ts_in_window                                       top_messages
0  frontend-0                         681     2022-03-20 14:30:00 CST   2022-03-20 14:59:56 CST  severity: warning, message: failed to retrieve...
1  frontend-1                         542     2022-03-20 14:30:02 CST   2022-03-20 14:59:57 CST  severity: warning, message: failed to retrieve...
2  frontend-2                         692     2022-03-20 14:30:01 CST   2022-03-20 14:59:58 CST  severity: warning, message: failed to retrieve...,       cmdb_id         log_name  total_points_in_window  sum_value_in_window  max_in_window earliest_ts_where_value_gt_0
0  frontend-0  log.error_count                      30                681.0           28.0      2022-03-20 14:30:00 CST
1  frontend-0    log.row_count                      30              52262.0         1917.0      2022-03-20 14:30:00 CST
2  frontend-1  log.error_count                      30                542.0           22.0      2022-03-20 14:30:00 CST
3  frontend-1    log.row_count                      30              42253.0         1618.0      2022-03-20 14:30:00 CST
4  frontend-2  log.error_count                      30                692.0           27.0      2022-03-20 14:30:00 CST
5  frontend-2    log.row_count                      30              53281.0         1994.0      2022-03-20 14:30:00 CST, 'Highest error_logs count: frontend-2 (692 errors); earliest error overall: 2022-03-20 14:30:00 CST')```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# reuse existing metric DataFrame and tz if available
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure types
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce').astype('Int64')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Targets
components = ['frontend-0','frontend-1','frontend-2','frontend2-0']
keywords = ['cpu','memory','mem','network','net','packet','transmit','receive','error','err',
            'fs','io','read','write','disk','throttled','file_descriptors','inodes']
pattern = '|'.join(keywords)

# Filter mask for component and kpi_name containing any keyword (case-insensitive)
mask_components = metric['cmdb_id'].isin(components)
mask_kpi = metric['kpi_name'].str.contains(pattern, case=False, na=False)

# Compute global thresholds (P95 and P5) using the full available series for each (cmdb_id,kpi_name)
grp_all = metric[mask_components & mask_kpi].groupby(['cmdb_id','kpi_name'])['value']
global_p95 = grp_all.quantile(0.95).rename('global_p95')
global_p05 = grp_all.quantile(0.05).rename('global_p5')
thresholds = pd.concat([global_p95, global_p05], axis=1).reset_index()

# Incident window (UTC+8): 2022-03-20 14:30:00 to 15:00:00 (start inclusive, end exclusive)
start_epoch = int(pd.Timestamp('2022-03-20 14:30:00', tz=tz).timestamp())
end_epoch   = int(pd.Timestamp('2022-03-20 15:00:00', tz=tz).timestamp())

# Rows in the incident window for the selected components & kpis
window_df = metric[mask_components & mask_kpi & (metric['timestamp'] >= start_epoch) & (metric['timestamp'] < end_epoch)].copy()

# Merge thresholds into window rows to evaluate anomalies; also keep thresholds-only rows (no window points)
win = window_df.merge(thresholds, on=['cmdb_id','kpi_name'], how='right')

# Mark anomaly flags based on global thresholds
win['anomaly_high'] = (win['value'] >= win['global_p95']).fillna(False)
win['anomaly_low']  = (win['value'] <= win['global_p5']).fillna(False)

# Aggregate in-window stats per (cmdb_id,kpi_name)
agg_counts = win.groupby(['cmdb_id','kpi_name']).agg(
    points_in_window = ('value', lambda s: int(s.count())),        # count non-NA values in window
    anomalies_high    = ('anomaly_high', 'sum'),
    anomalies_low     = ('anomaly_low', 'sum'),
    max_in_window     = ('value', 'max'),
    min_in_window     = ('value', 'min')
).reset_index()

# Earliest anomaly timestamps for high and low separately (compute from window_df merged with thresholds)
# Use window_df merged left to ensure timestamps correspond to actual window rows
win_rows = window_df.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
win_rows['anomaly_high'] = (win_rows['value'] >= win_rows['global_p95']).fillna(False)
win_rows['anomaly_low']  = (win_rows['value'] <= win_rows['global_p5']).fillna(False)

high_ts = win_rows[win_rows['anomaly_high']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().rename('earliest_anomaly_high_ts')
low_ts  = win_rows[win_rows['anomaly_low']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().rename('earliest_anomaly_low_ts')

# Combine thresholds with counts and earliest timestamps
result = thresholds.merge(agg_counts, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(high_ts.reset_index(), on=['cmdb_id','kpi_name'], how='left')
result = result.merge(low_ts.reset_index(), on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for numeric aggregates
result['points_in_window'] = result['points_in_window'].fillna(0).astype(int)
result['anomalies_high'] = result['anomalies_high'].fillna(0).astype(int)
result['anomalies_low'] = result['anomalies_low'].fillna(0).astype(int)

# Format earliest timestamps to UTC+8 strings or None
def epoch_to_local_str(x):
    if pd.isna(x):
        return None
    try:
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return None

result['earliest_anomaly_high_ts'] = result['earliest_anomaly_high_ts'].apply(epoch_to_local_str)
result['earliest_anomaly_low_ts']  = result['earliest_anomaly_low_ts'].apply(epoch_to_local_str)

# Round numeric thresholds and max/min for compactness
result['global_p95'] = result['global_p95'].round(6)
result['global_p5']  = result['global_p5'].round(6)
result['max_in_window'] = result['max_in_window'].round(6)
result['min_in_window'] = result['min_in_window'].round(6)

# Prepare final columns and sort by total anomalies desc, limit top 20
result['total_anoms'] = result['anomalies_high'] + result['anomalies_low']
out_cols = ['cmdb_id','kpi_name','global_p95','global_p5','points_in_window',
            'anomalies_high','anomalies_low','earliest_anomaly_high_ts','earliest_anomaly_low_ts',
            'max_in_window','min_in_window','total_anoms']
result_table = result[out_cols].sort_values('total_anoms', ascending=False).head(20).reset_index(drop=True).drop(columns=['total_anoms'])

# Summary: total anomaly points across selected frontend resource KPIs in-window and earliest anomaly timestamp among them
total_anomalies = int(result['anomalies_high'].sum() + result['anomalies_low'].sum())
# earliest among all anomaly timestamps (consider both high and low)
all_earliest_epochs = []
if result['earliest_anomaly_high_ts'].notna().any():
    # convert back to epochs for comparison
    for s in result['earliest_anomaly_high_ts'].dropna():
        try:
            all_earliest_epochs.append(int(pd.to_datetime(s).tz_localize(tz).tz_convert('UTC').timestamp()))
        except Exception:
            pass
if result['earliest_anomaly_low_ts'].notna().any():
    for s in result['earliest_anomaly_low_ts'].dropna():
        try:
            all_earliest_epochs.append(int(pd.to_datetime(s).tz_localize(tz).tz_convert('UTC').timestamp()))
        except Exception:
            pass

if all_earliest_epochs:
    earliest_overall = pd.to_datetime(min(all_earliest_epochs), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    earliest_overall = None

summary_line = f"Total anomaly points (selected frontend resource KPIs) in window: {total_anomalies}; earliest anomaly: {earliest_overall}"

# Compact display
result_table, total_anomalies, earliest_overall, summary_line

```
Out[5]:
```
Summary (plain English, times in UTC+8):

- What I did: looked at metric KPIs related to CPU, memory, network, packets, fs/io, throttling, file_descriptors, inodes, etc., for frontend-0, frontend-1, frontend-2, frontend2-0. I computed global P95 and P5 from the full series, then evaluated the 2022-03-20 14:30:00–15:00:00 window for anomalies (value >= P95 or <= P5).

- Key numbers:
  - Total anomaly points across the selected frontend resource/network/fs KPIs in the window: 19,740.
  - Earliest anomaly observed: 2022-03-20 14:30:00 (UTC+8).

- Main findings:
  - Many frontend KPI series are degenerate (global P95 and P5 equal 0.0 or another single constant). As a result, every point in the 30-minute window (30 points per metric) meets the anomaly rule (both high and low), so most listed KPI rows report 30 points_in_window and 30 anomalies.
  - Top flagged KPIs include numerous mesh/istio request/response/error metrics and container FS/IO and CPU-related metrics (e.g., container_file_descriptors, container_fs_io_*, container_cpu_*), often with global_p95 = global_p5 = 0.0 (or a single constant like 22.0 for file_descriptors).
  - Earliest anomalies for the top rows are at the window start (14:30:00), and in-window max/min values are typically the constant value (often 0.0).

- Interpretation: the percentile-based rule (value >= P95 or <= P5) generates large counts when the metric distributions are constant or mostly zero; this likely produces many misleading/ambiguous anomaly flags rather than meaningful resource spikes.

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                           kpi_name  global_p95  global_p5  points_in_window  anomalies_high  anomalies_low earliest_anomaly_high_ts  earliest_anomaly_low_ts  max_in_window  min_in_window
0   frontend2-0  mesh.source.frontend2.unknown.istio_response_m...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
1   frontend2-0  mesh.source.frontend2.unknown.istio_response_b...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
2    frontend-0  container.node-5.container_cpu_cfs_throttled_p...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
3    frontend-0  container.node-5.container_cpu_cfs_throttled_s...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
4    frontend-0    container.node-5.container_cpu_load_average_10s         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
5   frontend2-0  mesh.source.frontend2.unknown.istio_requests.g...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
6   frontend2-0  mesh.source.frontend2.unknown.istio_request_me...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
7   frontend2-0  mesh.source.frontend2.unknown.istio_request_du...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
8    frontend-0        container.node-5.container_file_descriptors        22.0       22.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST           22.0           22.0
9    frontend-0     container.node-5.container_fs_inodes./dev/vda1         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
10   frontend-0  container.node-5.container_fs_inodes_free./dev...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
11   frontend-0  container.node-5.container_fs_io_current./dev/...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
12   frontend-0  container.node-5.container_fs_io_time_seconds....         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
13   frontend-0  container.node-5.container_fs_io_time_weighted...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
14  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
15  frontend2-0  mesh.source.frontend2.cartservice2.istio_reque...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
16  frontend2-0  mesh.source.frontend2.cartservice2.istio_respo...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
17  frontend2-0  mesh.source.frontend2.cartservice2.istio_respo...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
18  frontend2-0  mesh.source.frontend2.adservice2.istio_respons...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0
19  frontend2-0  mesh.source.frontend2.adservice2.istio_respons...         0.0        0.0                30              30             30  2022-03-20 14:30:00 CST  2022-03-20 14:30:00 CST            0.0            0.0, 19740, '2022-03-20 14:30:00 CST', 'Total anomaly points (selected frontend resource KPIs) in window: 19740; earliest anomaly: 2022-03-20 14:30:00 CST')```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# reuse existing error_logs if present, otherwise load
try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')

tz = pytz.timezone('Asia/Shanghai')

# ensure types
error_logs['timestamp'] = pd.to_numeric(error_logs['timestamp'], errors='coerce').astype('Int64')

# define frontends and window (UTC+8)
frontends = ['frontend-0','frontend-1','frontend-2']
start_epoch = int(pd.Timestamp('2022-03-20 14:30:00', tz=tz).timestamp())
end_epoch   = int(pd.Timestamp('2022-03-20 14:35:00', tz=tz).timestamp())

# filter rows
slice_df = error_logs[
    (error_logs['cmdb_id'].isin(frontends)) &
    (error_logs['timestamp'] >= start_epoch) &
    (error_logs['timestamp'] < end_epoch)
].copy()

# convert timestamp to readable UTC+8 string for output
slice_df['timestamp'] = pd.to_datetime(slice_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# ensure full message is not truncated when displayed
pd.set_option('display.max_colwidth', None)

# prepare first up to 20 rows sorted by timestamp (ascending) with required columns
slice_df_display = slice_df.sort_values('timestamp').loc[:, ['timestamp','cmdb_id','message']].head(20).reset_index(drop=True)

# compact count of distinct full messages (top 5)
msg_counts = (slice_df.groupby('message').size().reset_index(name='count').sort_values('count', ascending=False).head(5).reset_index(drop=True))
top_messages_df = msg_counts

# Display compact outputs
slice_df_display, top_messages_df

```
Out[6]:
```
Summary (plain English, times in UTC+8):

Time window: 2022-03-20 14:30:00 – 14:35:00 (UTC+8), frontends: frontend-0, frontend-1, frontend-2.

1) First up to 20 rows (sorted by timestamp) — each row shows timestamp, cmdb_id, full message:
- 2022-03-20 14:30:00 CST — frontend-0 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:01 CST — frontend-2 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:02 CST — frontend-1 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:04 CST — frontend-0 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:04 CST — frontend-2 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:05 CST — frontend-0 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:05 CST — frontend-1 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:07 CST — frontend-2 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:07 CST — frontend-1 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:08 CST — frontend-0 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:10 CST — frontend-0 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:10 CST — frontend-1 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:11 CST — frontend-0 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:11 CST — frontend-2 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:13 CST — frontend-2 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:14 CST — frontend-0 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:15 CST — frontend-0 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:15 CST — frontend-1 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:16 CST — frontend-0 — severity: warning, message: failed to retrieve ads
- 2022-03-20 14:30:17 CST — frontend-2 — severity: warning, message: failed to retrieve ads

2) Distinct full messages in this 5-minute slice (top 5):
- "severity: warning, message: failed to retrieve ads" — 312 occurrences

Interpretation: the 5-minute slice contains a single repeated error message ("failed to retrieve ads") across all three frontends, occurring hundreds of times (312) in that interval.

The original code execution output of IPython Kernel is also provided below for reference:

(                  timestamp     cmdb_id                                             message
0   2022-03-20 14:30:00 CST  frontend-0  severity: warning, message: failed to retrieve ads
1   2022-03-20 14:30:01 CST  frontend-2  severity: warning, message: failed to retrieve ads
2   2022-03-20 14:30:02 CST  frontend-1  severity: warning, message: failed to retrieve ads
3   2022-03-20 14:30:04 CST  frontend-0  severity: warning, message: failed to retrieve ads
4   2022-03-20 14:30:04 CST  frontend-2  severity: warning, message: failed to retrieve ads
5   2022-03-20 14:30:05 CST  frontend-0  severity: warning, message: failed to retrieve ads
6   2022-03-20 14:30:05 CST  frontend-1  severity: warning, message: failed to retrieve ads
7   2022-03-20 14:30:07 CST  frontend-2  severity: warning, message: failed to retrieve ads
8   2022-03-20 14:30:07 CST  frontend-1  severity: warning, message: failed to retrieve ads
9   2022-03-20 14:30:08 CST  frontend-0  severity: warning, message: failed to retrieve ads
10  2022-03-20 14:30:10 CST  frontend-0  severity: warning, message: failed to retrieve ads
11  2022-03-20 14:30:10 CST  frontend-1  severity: warning, message: failed to retrieve ads
12  2022-03-20 14:30:11 CST  frontend-0  severity: warning, message: failed to retrieve ads
13  2022-03-20 14:30:11 CST  frontend-2  severity: warning, message: failed to retrieve ads
14  2022-03-20 14:30:13 CST  frontend-2  severity: warning, message: failed to retrieve ads
15  2022-03-20 14:30:14 CST  frontend-0  severity: warning, message: failed to retrieve ads
16  2022-03-20 14:30:15 CST  frontend-0  severity: warning, message: failed to retrieve ads
17  2022-03-20 14:30:15 CST  frontend-1  severity: warning, message: failed to retrieve ads
18  2022-03-20 14:30:16 CST  frontend-0  severity: warning, message: failed to retrieve ads
19  2022-03-20 14:30:17 CST  frontend-2  severity: warning, message: failed to retrieve ads,                                               message  count
0  severity: warning, message: failed to retrieve ads    312)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# reuse existing 'trace' and 'tz' if available
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure types
trace['timestamp'] = pd.to_numeric(trace['timestamp'], errors='coerce').astype('Int64')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# Filter: trace_name contains 'adservice' (case-insensitive) and within incident window UTC+8
mask_name = trace['trace_name'].str.contains('adservice', case=False, na=False)
start_epoch = int(pd.Timestamp('2022-03-20 14:30:00', tz=tz).timestamp())
end_epoch   = int(pd.Timestamp('2022-03-20 15:00:00', tz=tz).timestamp())

win = trace[mask_name & (trace['timestamp'] >= start_epoch) & (trace['timestamp'] < end_epoch)].copy()

# If empty, return empty structures
if win.empty:
    result_table = pd.DataFrame(columns=[
        'cmdb_id','trace_name','points_in_window','sum_error_rate_in_window','max_error_rate_in_window',
        'sum_row_count_in_window','min_row_count_in_window','earliest_ts_error_rate_gt_0','earliest_ts_row_count_eq_0'
    ])
    total_sum_error_rate = 0
    earliest_overall = None
    summary_line = "No adservice-related traces in the window."
else:
    # Basic grouped aggregates
    grouped = win.groupby(['cmdb_id','trace_name']).agg(
        points_in_window = ('value','count'),
        sum_val = ('value','sum'),
        max_val = ('value','max'),
        min_val = ('value','min')
    ).reset_index()

    # Flags at group level
    grouped['is_error_rate'] = grouped['trace_name'].str.contains('error_rate', case=False, na=False)
    grouped['is_row_count']  = grouped['trace_name'].str.contains('row_count', case=False, na=False)

    # earliest timestamp where error_rate > 0 per group
    err_pos = win[win['trace_name'].str.contains('error_rate', case=False, na=False) & (win['value'] > 0)]
    err_first = err_pos.groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index(name='earliest_ts_error_rate_gt_0')

    # earliest timestamp where row_count == 0 per group
    row_zero = win[win['trace_name'].str.contains('row_count', case=False, na=False) & (win['value'] == 0)]
    row_zero_first = row_zero.groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index(name='earliest_ts_row_count_eq_0')

    # Merge earliest timestamps
    out = grouped.merge(err_first, on=['cmdb_id','trace_name'], how='left')
    out = out.merge(row_zero_first, on=['cmdb_id','trace_name'], how='left')

    # Compute requested columns
    out['sum_error_rate_in_window'] = out.apply(lambda r: float(r['sum_val']) if r['is_error_rate'] else 0.0, axis=1)
    out['max_error_rate_in_window'] = out.apply(lambda r: float(r['max_val']) if r['is_error_rate'] else None, axis=1)
    out['sum_row_count_in_window']  = out.apply(lambda r: float(r['sum_val']) if r['is_row_count'] else 0.0, axis=1)
    out['min_row_count_in_window']  = out.apply(lambda r: float(r['min_val']) if r['is_row_count'] else None, axis=1)

    # Format earliest timestamp columns to UTC+8 strings or None
    def epoch_to_local_str(x):
        if pd.isna(x):
            return None
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    out['earliest_ts_error_rate_gt_0'] = out['earliest_ts_error_rate_gt_0'].apply(epoch_to_local_str)
    out['earliest_ts_row_count_eq_0']  = out['earliest_ts_row_count_eq_0'].apply(epoch_to_local_str)

    # Select final columns
    result_table = out[[
        'cmdb_id','trace_name','points_in_window',
        'sum_error_rate_in_window','max_error_rate_in_window',
        'sum_row_count_in_window','min_row_count_in_window',
        'earliest_ts_error_rate_gt_0','earliest_ts_row_count_eq_0'
    ]].copy()

    # Sort by sum_error_rate_in_window desc and limit to top 20
    result_table = result_table.sort_values('sum_error_rate_in_window', ascending=False).head(20).reset_index(drop=True)

    # Summary totals
    total_sum_error_rate = float(result_table['sum_error_rate_in_window'].sum())
    # earliest overall error_rate>0 across all adservice traces in window
    all_err_epochs = err_first['earliest_ts_error_rate_gt_0']
    if not all_err_epochs.empty:
        earliest_overall = epoch_to_local_str(int(all_err_epochs.min()))
    else:
        earliest_overall = None

    summary_line = f"Total sum_error_rate (adservice traces) in-window: {total_sum_error_rate}; earliest error_rate>0: {earliest_overall}"

# Compact outputs
result_table, total_sum_error_rate, earliest_overall, summary_line

```
Out[7]:
```
Summary (plain English, times in UTC+8):

- What I checked: adservice-related traces (trace_name contains "adservice") for the 30-minute incident window 2022-03-20 14:30:00–15:00:00 (UTC+8).

- Main findings (compact):
  - Every reported adservice-related trace row in the window has 30 points (one per minute).
  - sum_error_rate_in_window = 0.0 for all listed error_rate traces; max_error_rate_in_window = 0.0. In other words, no adservice trace reported error_rate > 0 during the window.
  - Row-count traces show nonzero traffic counts:
    - Examples: frontend-0 → trace.to_adservice-0.row_count: sum = 681, min = 17
      frontend-0 → trace.to_adservice-2.row_count: sum = 682, min = 18
      frontend-1 → trace.to_adservice-0.row_count: sum = 542, min = 14
      frontend-1 → trace.to_adservice-1.row_count: sum = 543, min = 14
  - earliest_ts_error_rate_gt_0: None (no in-window timestamp with error_rate>0).
  - earliest_ts_row_count_eq_0: None for the shown groups (no minute with row_count == 0 in the listed groups).

- Summary line:
  - Total sum_error_rate across all adservice-related traces in-window: 0.0.
  - Earliest timestamp with any adservice trace having error_rate>0: none (no such occurrences).

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                          trace_name  points_in_window  sum_error_rate_in_window  max_error_rate_in_window  sum_row_count_in_window  min_row_count_in_window earliest_ts_error_rate_gt_0 earliest_ts_row_count_eq_0
0   frontend-0  trace.to_adservice-0.duration_mean                30                       0.0                       NaN                      0.0                      NaN                        None                       None
1   frontend-0   trace.to_adservice-0.duration_p95                30                       0.0                       NaN                      0.0                      NaN                        None                       None
2   frontend-0     trace.to_adservice-0.error_rate                30                       0.0                       0.0                      0.0                      NaN                        None                       None
3   frontend-0      trace.to_adservice-0.row_count                30                       0.0                       NaN                    681.0                     17.0                        None                       None
4   frontend-0  trace.to_adservice-1.duration_mean                30                       0.0                       NaN                      0.0                      NaN                        None                       None
5   frontend-0   trace.to_adservice-1.duration_p95                30                       0.0                       NaN                      0.0                      NaN                        None                       None
6   frontend-0     trace.to_adservice-1.error_rate                30                       0.0                       0.0                      0.0                      NaN                        None                       None
7   frontend-0      trace.to_adservice-1.row_count                30                       0.0                       NaN                    681.0                     17.0                        None                       None
8   frontend-0  trace.to_adservice-2.duration_mean                30                       0.0                       NaN                      0.0                      NaN                        None                       None
9   frontend-0   trace.to_adservice-2.duration_p95                30                       0.0                       NaN                      0.0                      NaN                        None                       None
10  frontend-0     trace.to_adservice-2.error_rate                30                       0.0                       0.0                      0.0                      NaN                        None                       None
11  frontend-0      trace.to_adservice-2.row_count                30                       0.0                       NaN                    682.0                     18.0                        None                       None
12  frontend-1  trace.to_adservice-0.duration_mean                30                       0.0                       NaN                      0.0                      NaN                        None                       None
13  frontend-1   trace.to_adservice-0.duration_p95                30                       0.0                       NaN                      0.0                      NaN                        None                       None
14  frontend-1     trace.to_adservice-0.error_rate                30                       0.0                       0.0                      0.0                      NaN                        None                       None
15  frontend-1      trace.to_adservice-0.row_count                30                       0.0                       NaN                    542.0                     14.0                        None                       None
16  frontend-1  trace.to_adservice-1.duration_mean                30                       0.0                       NaN                      0.0                      NaN                        None                       None
17  frontend-1   trace.to_adservice-1.duration_p95                30                       0.0                       NaN                      0.0                      NaN                        None                       None
18  frontend-1     trace.to_adservice-1.error_rate                30                       0.0                       0.0                      0.0                      NaN                        None                       None
19  frontend-1      trace.to_adservice-1.row_count                30                       0.0                       NaN                    543.0                     14.0                        None                       None, 0.0, None, 'Total sum_error_rate (adservice traces) in-window: 0.0; earliest error_rate>0: None')```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# reuse existing dataframes if present, otherwise load
try:
    log
except NameError:
    log = pd.read_csv('log.csv')
try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure types
log['timestamp'] = pd.to_numeric(log['timestamp'], errors='coerce').astype('Int64')
log['value'] = pd.to_numeric(log['value'], errors='coerce')
error_logs['timestamp'] = pd.to_numeric(error_logs['timestamp'], errors='coerce').astype('Int64')

# Targets and incident window (UTC+8)
ad_pods = ['adservice-0','adservice-1','adservice-2','adservice2-0']
start_epoch = int(pd.Timestamp('2022-03-20 14:30:00', tz=tz).timestamp())
end_epoch   = int(pd.Timestamp('2022-03-20 15:00:00', tz=tz).timestamp())

# 1) log.csv aggregates for the adservice pods within window
log_win = log[
    (log['cmdb_id'].isin(ad_pods)) &
    (log['timestamp'] >= start_epoch) &
    (log['timestamp'] < end_epoch)
].copy()

if not log_win.empty:
    # points_in_window, sum_value_in_window, max_in_window
    agg = log_win.groupby(['cmdb_id','log_name']).agg(
        points_in_window = ('value','count'),
        sum_value_in_window = ('value','sum'),
        max_in_window = ('value','max')
    ).reset_index()

    # earliest_ts_where_value_gt_0
    pos = log_win[log_win['value'] > 0].copy()
    if not pos.empty:
        earliest_pos = pos.groupby(['cmdb_id','log_name'])['timestamp'].min().reset_index(name='earliest_ts_where_value_gt_0')
    else:
        earliest_pos = pd.DataFrame(columns=['cmdb_id','log_name','earliest_ts_where_value_gt_0'])

    log_agg_df = agg.merge(earliest_pos, on=['cmdb_id','log_name'], how='left')

    # format earliest timestamp to local string or None
    def epoch_to_local_str(x):
        if pd.isna(x):
            return None
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    log_agg_df['earliest_ts_where_value_gt_0'] = log_agg_df['earliest_ts_where_value_gt_0'].apply(lambda x: epoch_to_local_str(x) if pd.notna(x) else None)

    # Round numeric sums for compactness
    log_agg_df['sum_value_in_window'] = log_agg_df['sum_value_in_window'].round(6)
    log_agg_df['max_in_window'] = log_agg_df['max_in_window'].round(6)

    # Keep only rows for the four ad pods (should already be) and present one row per cmdb_id/log_name
    log_agg_df = log_agg_df[['cmdb_id','log_name','points_in_window','sum_value_in_window','max_in_window','earliest_ts_where_value_gt_0']].reset_index(drop=True)
else:
    log_agg_df = pd.DataFrame(columns=['cmdb_id','log_name','points_in_window','sum_value_in_window','max_in_window','earliest_ts_where_value_gt_0'])

# 2) error_logs.csv: check for any rows for these adservice cmdb_ids in the window
err_win = error_logs[
    (error_logs['cmdb_id'].isin(ad_pods)) &
    (error_logs['timestamp'] >= start_epoch) &
    (error_logs['timestamp'] < end_epoch)
].copy()

if not err_win.empty:
    # Convert timestamp to local string for display and return up to first 10 rows sorted by timestamp
    err_win_sorted = err_win.sort_values('timestamp').head(10).copy()
    err_win_sorted['timestamp'] = pd.to_datetime(err_win_sorted['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    err_rows_df = err_win_sorted[['timestamp','cmdb_id','message']].reset_index(drop=True)
    any_error_logs = True
    earliest_err_epoch = err_win['timestamp'].min()
    earliest_err_ts = pd.to_datetime(int(earliest_err_epoch), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    err_rows_df = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
    any_error_logs = False
    earliest_err_ts = None

# One-line summary: whether adservice pods recorded errors (in error_logs) and earliest error timestamp if present
summary_line = f"Adservice error_logs present: {'Yes' if any_error_logs else 'No'}; earliest error_logs timestamp: {earliest_err_ts}"

# Output compact results
log_agg_df, err_rows_df, summary_line

```
Out[8]:
```
Summary (plain English, times in UTC+8):

Log aggregates (window 2022-03-20 14:30:00–15:00:00):

- adservice-0
  - log.error_count: 30 points, sum = 0.0, max = 0.0, no timestamp with value>0
  - log.row_count:   30 points, sum = 42,496.0, max = 1,552.0, earliest value>0 = 2022-03-20 14:30:00

- adservice-1
  - log.error_count: 30 points, sum = 0.0, max = 0.0, no timestamp with value>0
  - log.row_count:   30 points, sum = 3,838.0,   max = 142.0,   earliest value>0 = 2022-03-20 14:30:00

- adservice-2
  - log.error_count: 30 points, sum = 0.0, max = 0.0, no timestamp with value>0
  - log.row_count:   30 points, sum = 3,836.0,   max = 144.0,   earliest value>0 = 2022-03-20 14:30:00

Error logs (error_logs.csv) for adservice pods in the window:
- No error_log rows found for adservice-0, adservice-1, adservice-2, or adservice2-0 in the incident window.

One-line summary:
- Adservice pods did NOT record errors in error_logs during the window; log.error_count metrics are all zero. Earliest adservice-side nonzero log activity (row_count>0) appears at 2022-03-20 14:30:00; there is no adservice error timestamp.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id         log_name  points_in_window  sum_value_in_window  max_in_window earliest_ts_where_value_gt_0
0  adservice-0  log.error_count                30                  0.0            0.0                         None
1  adservice-0    log.row_count                30              42496.0         1552.0      2022-03-20 14:30:00 CST
2  adservice-1  log.error_count                30                  0.0            0.0                         None
3  adservice-1    log.row_count                30               3838.0          142.0      2022-03-20 14:30:00 CST
4  adservice-2  log.error_count                30                  0.0            0.0                         None
5  adservice-2    log.row_count                30               3836.0          144.0      2022-03-20 14:30:00 CST, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], 'Adservice error_logs present: No; earliest error_logs timestamp: None')```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# reuse existing 'trace' and 'tz' if available, otherwise load/define
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure correct dtypes
trace['timestamp'] = pd.to_numeric(trace['timestamp'], errors='coerce').astype('Int64')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# Filter trace_name for adservice AND error_rate (case-insensitive) using full series (no time filter)
mask_ad_err = trace['trace_name'].str.contains('adservice', case=False, na=False) & trace['trace_name'].str.contains('error_rate', case=False, na=False)
ad_err_df = trace[mask_ad_err].copy()

# Group by (cmdb_id, trace_name) and compute totals and nonzero counts
grp = ad_err_df.groupby(['cmdb_id','trace_name'])
summary = grp.agg(
    total_rows = ('value','count'),
    count_error_rate_nonzero = ('value', lambda s: int((s>0).sum()))
).reset_index()

# Earliest timestamp with value>0 per group
pos = ad_err_df[ad_err_df['value']>0].copy()
if not pos.empty:
    earliest_pos = pos.groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index(name='earliest_epoch')
    # convert epoch -> localized string
    earliest_pos['earliest_ts_error_rate_gt_0'] = pd.to_datetime(earliest_pos['earliest_epoch'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    earliest_pos = earliest_pos.drop(columns=['earliest_epoch'])
else:
    earliest_pos = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_ts_error_rate_gt_0'])

# Combine
result = summary.merge(earliest_pos, on=['cmdb_id','trace_name'], how='left')

# Fill NaN earliest with None for compactness
result['earliest_ts_error_rate_gt_0'] = result['earliest_ts_error_rate_gt_0'].where(result['earliest_ts_error_rate_gt_0'].notna(), None)

# Sort by count_error_rate_nonzero desc and limit to top 20
result_table = result.sort_values('count_error_rate_nonzero', ascending=False).head(20).reset_index(drop=True)

# One-line summary: total nonzero count across all adservice-related error_rate traces and earliest timestamp among them
total_nonzero = int(result['count_error_rate_nonzero'].sum())
if not pos.empty:
    earliest_overall_epoch = int(pos['timestamp'].min())
    earliest_overall = pd.to_datetime(earliest_overall_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    earliest_overall = None

summary_line = f"Total nonzero adservice error_rate rows: {total_nonzero}; earliest nonzero error_rate: {earliest_overall}"

# Compact outputs
result_table, total_nonzero, earliest_overall, summary_line

```
Out[9]:
```
Summary (plain English):

- I scanned all trace series whose name contains both "adservice" and "error_rate" across the full dataset.

- For each (cmdb_id, trace_name) checked:
  - total rows available (full series): 30 per trace,
  - count of rows with error_rate > 0: 0 for every trace,
  - earliest timestamp with error_rate > 0: none.

- Result: No adservice-related trace reported any nonzero error_rate. Total nonzero error_rate rows across all adservice traces = 0; there is no earliest nonzero error_rate timestamp.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                        trace_name  total_rows  count_error_rate_nonzero earliest_ts_error_rate_gt_0
0   frontend-0   trace.to_adservice-0.error_rate          30                         0                        None
1   frontend-0   trace.to_adservice-1.error_rate          30                         0                        None
2   frontend-0   trace.to_adservice-2.error_rate          30                         0                        None
3   frontend-1   trace.to_adservice-0.error_rate          30                         0                        None
4   frontend-1   trace.to_adservice-1.error_rate          30                         0                        None
5   frontend-1   trace.to_adservice-2.error_rate          30                         0                        None
6   frontend-2   trace.to_adservice-0.error_rate          30                         0                        None
7   frontend-2   trace.to_adservice-1.error_rate          30                         0                        None
8   frontend-2   trace.to_adservice-2.error_rate          30                         0                        None
9  frontend2-0  trace.to_adservice2-0.error_rate          30                         0                        None, 0, None, 'Total nonzero adservice error_rate rows: 0; earliest nonzero error_rate: None')```
```